## Importing libraries

In [ ]:
import foldersfiles as ff
import functions as fn

import numpy as np
import pandas as pd
import math

import json

from pathlib import Path

import pickle

## Importing data

In [ ]:
df = fn.import_func(
    ff.CLEANED_PROCESSED_DATA_DIR / ff.PARQUETS / f'{ff.EUMACO}-simple.parquet'
)

In [ ]:
df_hospit = pd.read_parquet(ff.PROCESSED_DATA_DIR / ff.PARQUETS / 'hosp-mid.parquet')

# with open(ff.PROCESSED_DATA_DIR / ff.PICKLES / f'{ff.EUMACO}-all.pickle', 'rb') as file:
# 	df = pickle.load(file)
print(df_hospit.iloc[[203, 204, 205]])
df_hospit.drop(204, inplace=True)
print(df_hospit.iloc[[203, 204, 205]])

## Analysis

In [ ]:
from IPython.display import display
from statsmodels.duration.hazard_regression import PHReg

# REQUIRED: replace None with the end of reliable hospitalization coverage.
# Example format: "2025-12-31 23:59:59"
FOLLOWUP_END = "2025-12-31 23:59:59"

# Optional: patient-specific end of observation, including death or loss
# of hospitalization coverage. Values must be datetime-compatible.
#
# Example:
# patient_end = pd.Series(
#     {"patient_001": "2025-06-15", "patient_002": "2025-09-20"}
# )
patient_end = None

LOOKBACK_DAYS = 7  # Illustrative choice; justify for your study.
MAX_ORDER = 3     # Analyze up to the third readmission; inspect counts below.

# Use one heart-rate source initially. Change this if appropriate.
HR_COLUMN = "spo2_hr_bpm"

FEATURES = {
    "weight_kg": "weight",
    "spo2_pct": "spo2",
    "syst_bp_mmhg": "sbp",
    HR_COLUMN: "hr",
}

if FOLLOWUP_END is None:
    raise ValueError("Set FOLLOWUP_END before continuing.")

followup_end = pd.Timestamp(FOLLOWUP_END)

In [ ]:
check = df_hospit[
    ["pseudopaz", "data_apertura", "data_chiusura"]
].copy()

check["admission_parsed"] = pd.to_datetime(
    check["data_apertura"], errors="raise"
)
check["discharge_parsed"] = pd.to_datetime(
    check["data_chiusura"], errors="raise"
)

bad = (
    check["discharge_parsed"].notna()
    & (check["discharge_parsed"] < check["admission_parsed"])
)

print(f"Rows with discharge before admission: {bad.sum()}")
display(check.loc[bad].head(30))

In [ ]:
# For strings such as "31/12/2024"
h["data_apertura"] = pd.to_datetime(
    h["data_apertura"],
    format="%d/%m/%Y",
    errors="raise",
)

h["data_chiusura"] = pd.to_datetime(
    h["data_chiusura"],
    format="%d/%m/%Y",
    errors="raise",
)

In [ ]:
h = df_hospit[
    ["pseudopaz", "data_apertura", "data_chiusura"]
].copy()

m = df_medical[
    ["pseudopaz", "datetime", *FEATURES.keys()]
].copy()

for frame in (h, m):
    if frame["pseudopaz"].isna().any():
        raise ValueError("Missing patient IDs must be resolved.")
    frame["pseudopaz"] = frame["pseudopaz"].astype(str)

# These calls assume dates are unambiguous and use consistent time zones.
# Supply format=... if your source uses a specific date format.
h["data_apertura"] = pd.to_datetime(h["data_apertura"], errors="raise")
h["data_chiusura"] = pd.to_datetime(h["data_chiusura"], errors="raise")
m["datetime"] = pd.to_datetime(m["datetime"], errors="raise")

if h["data_apertura"].isna().any() or m["datetime"].isna().any():
    raise ValueError("Missing admission or measurement timestamps.")

for source in FEATURES:
    m[source] = pd.to_numeric(m[source], errors="raise")
    m[source] = m[source].replace([np.inf, -np.inf], np.nan)

h = (
    h.drop_duplicates()
     .sort_values(["pseudopaz", "data_apertura"])
     .reset_index(drop=True)
)

bad_duration = (
    h["data_chiusura"].notna()
    & (h["data_chiusura"] < h["data_apertura"])
)
if bad_duration.any():
    display(h.loc[bad_duration])
    raise ValueError("Some discharges occur before admission.")

h["next_admission"] = (
    h.groupby("pseudopaz")["data_apertura"].shift(-1)
)

# An ongoing final hospitalization may have no discharge date.
# A missing discharge before another admission needs investigation.
bad_missing = (
    h["data_chiusura"].isna()
    & h["next_admission"].notna()
)

bad_overlap = (
    h["next_admission"].notna()
    & (h["next_admission"] <= h["data_chiusura"])
)

if (bad_missing | bad_overlap).any():
    display(h.loc[bad_missing | bad_overlap])
    raise ValueError(
        "Resolve missing discharges, overlaps and touching episodes first."
    )

# Conflicting measurements at exactly the same timestamp need a rule.
m = m.drop_duplicates()
if m.duplicated(["pseudopaz", "datetime"]).any():
    raise ValueError(
        "Multiple measurement rows share patient and timestamp. "
        "Resolve or aggregate these explicitly before continuing."
    )